<a href="https://colab.research.google.com/github/8Dis-like/UCLALearning/blob/main/Language_Modelling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Language Modeling and Transformers

The project will consist of two broad parts.

1. **Baseline Generative Language Model**: We will train a simple Bigram language model on the text data. We will use this model to generate a mini story.
2. **Implementing Mini GPT**: We will implement a mini version of the GPT model layer by layer and attempt to train it on the text data. You will then load pretrained weights provided and generate a mini story.

## Some general instructions

1. Please keep the name of layers consistent with what is requested in the `model.py` file for each layer, this helps us test in each function independently.
2. Please check to see if the bias is to be set to false or true for all linear layers (it is mentioned in the doc string)
3. As a general rule please read the docstring well, it contains information you will need to write the code.
4. All configs are defined in `config.py` for the first part. While you are writing the code, do not change the values in the config file since we use them to test. Once you have passed all the tests please feel free to vary the parameter as you please.
5. You will need to fill in `train.py` and run it to train the model. If you are running into memory issues please feel free to change the `batch_size` in the `config.py` file. If you are working on Colab please make sure to use the GPU runtime and feel free to copy over the training code to the notebook.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import torch
import tiktoken

In [ ]:
from model import BigramLanguageModel, SingleHeadAttention, MultiHeadAttention, FeedForwardLayer, LayerNorm, TransformerLayer, MiniGPT
from config import BigramConfig, MiniGPTConfig
import tests

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# If not provided, download from https://drive.google.com/file/d/1g09qUM9WibdfQVgkj6IAj8K2S3SGwc91/view?usp=sharing
path_to_bigram_tester = "./pretrained_models/bigram_tester.pt" # Load the bigram model with name bigram_tester.pt
path_to_gpt_tester = "./pretrained_models/minigpt_tester.pt" # Load the gpt model with name minigpt_tester.pt

##  Bigram Language Model (10 points)

A bigram language model is a type of probabilistic language model that predicts a word given the previous word in the sequence. The model is trained on a text corpus and learns the probability of a word given the previous word.



### Implement the Bigram model (5 points)

Please complete the `BigramLanguageModel` class in model.py. We will model a Bigram language model using a simple MLP with one hidden layer. The model will take in the previous word index and output the logits over the vocabulary for the next word.

In [ ]:
# Test implementation for Bigram Language Model
model = BigramLanguageModel(BigramConfig)
tests.check_bigram(model, path_to_bigram_tester, device)

'TEST CASE PASSED!!!'

### Training the Bigram Language Model (2.5 points)

Complete the code in `train.py` to train the Bigram language model on the text data. Please provide plots for both the training and validation in the cell below.

Some notes on the training process:

1. You should be able to train the model slowly on your local machine.
2. Training it on Colab will help with speed.
3.  <span style="color:red">To get full points for this section it is sufficient to show that the loss is decreasing over time</span>. You should see it saturate to a value close to around 5-6 but as long as you see it decreasing then saturating you should be good.
4. Please log the loss curves either on wandb, tensorboard or any other logger of your choice and please attach them below.

In [ ]:
from train import solver

In [ ]:
solver(model_name="bigram")

### Train and Valid Plots


** Show the training and validation loss plots **

### Generation (2.5 points)

Complete the code in the `generate` method of the Bigram class and generate a mini story using the trained Bigram language model. The model will take in the previous word index and output the next word index.

Start with the following seed sentence:
    
    `"once upon a time"`
    

In [ ]:
# TODO: Specify the path to your trained model
model_path = "/home/zugho/MiniGPTStudentVersion/models/bigram/mini_model_checkpoint_500000.pt"
model = BigramLanguageModel(BigramConfig)
tokenizer = tiktoken.get_encoding("gpt2")
model.load_state_dict(torch.load(model_path)["model_state_dict"])

<All keys matched successfully>

In [ ]:
model.to(device)
gen_sent = "Once upon a time"
gen_tokens = torch.tensor(tokenizer.encode(gen_sent)).unsqueeze(0)
print("Generating text starting with:", gen_tokens.shape)
gen_tokens = gen_tokens.to(device)
model.eval()
print(
    tokenizer.decode(
        model.generate(gen_tokens, max_new_tokens=200).squeeze().tolist()
    )
)

Generating text starting with: torch.Size([1, 4])
Once upon a time," Tom was a walk.
They swam and his hands too too fast, and they can be friends and saw a dinosaur. She went down the pan. The shore, a museum. Max came the purple pond," Lily. Do you good grew up.
The duckasis for his dad an Jazz was very happy." The old and he could not so soft came home.
Her handsThey crayons.
Anna. The end of two bigP stuck in the While said and made a Buzz pulling inside. Tim was too about the soap and eat the park and always ask for bikes.Once upon a fun. They looked over followed the dog had a distance. They became yummy, there was able to bus. One day, until he would be examine it away. One day, he made them in because it is nice. They both laughed and theLook forippin. They were very well. It thread. They went closer to get amazing sunny day fair and angry


### Observation and Analysis

Please answer the following questions.

1. What can we say about the generated text in terms of grammar and coherence?

The text is largely incoherent and lacks global grammatical structure. While it occasionally forms sensible local phrases of two or three words (e.g., "Once upon a time," "saw a dinosaur"), it rapidly devolves into nonsense and jarring transitions (e.g., "Her handsThey crayons"). There is no overarching narrative or logical flow.

2. What are the limitations of the Bigram language model?

The fundamental limitation of a Bigram model is its extremely limited context window. Because it only looks at the single preceding word to predict the next one ($P(x_t | x_{t-1})$), it has absolutely no "memory." It cannot track subjects, maintain a topic, or capture long-range dependencies required to form proper sentences or paragraphs.

3. If the model is scaled with more parameters do you expect the bigram model to get substantially better? Why or why not?

No, scaling the parameters will not make it substantially better. The bottleneck is the architecture, not the parameter count. Even with massive hidden layers, the model is still mathematically restricted to only "seeing" one word at a time. To generate coherent text, the model needs to process a longer sequence of past tokens (which is exactly what the MiniGPT section will solve using self-attention!).

## Mini GPT (90 points)

We will implement a decoder style transformer model like we discussed in lecture, which is a scaled down version of the [GPT model](https://cdn.openai.com/research-covers/language-unsupervised/language_understanding_paper.pdf).

All the model components follow directly from the original [Attention is All You Need](https://arxiv.org/abs/1706.03762) paper. The only difference is we will use prenormalization and learnt positional embeddings instead of fixed ones.

We will now implement each layer step by step checking if it is implemented correctly in the process. We will finally put together all our layers to get a fully fledged GPT model.

<span style="color:red">Later layers might depend on previous layers so please make sure to check the previous layers before moving on to the next one.</span>

### Single Head Causal Attention (20 points)

We will first implement the single head causal attention layer. This layer is the same as the scaled dot product attention layer but with a causal mask to prevent the model from looking into the future.

Recall that Each head has a Key, Query and Value Matrix and the scaled dot product attention is calculated as :

\begin{equation}
\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V
\end{equation}

where $d_k$ is the dimension of the key matrix.

Figure below from the original paper shows how the layer is to be implemented.

![image](./Images/Single_Head.png)

Image credits: [Attention is All You Need Paper](https://arxiv.org/abs/1706.03762)

Please complete the `SingleHeadAttention` class in `model.py`

In [ ]:
model = SingleHeadAttention(MiniGPTConfig.embed_dim, MiniGPTConfig.embed_dim//4, MiniGPTConfig.embed_dim//4) # configs are set as such for testing do not modify
tests.check_singleheadattention(model, path_to_gpt_tester, device)

'TEST CASE PASSED!!!'

### Multi Head Attention (10 points)

Now that we have a single head working, we will now scale this across multiple heads, remember that with multihead attention we compute perform head number of parallel attention operations. We then concatenate the outputs of these parallel attention operations and project them back to the desired dimension using an output linear layer.

Figure below from the original paper shows how the layer is to be implemented.

![image](./Images/MultiHead.png)

Image credits: [Attention is All You Need Paper](https://arxiv.org/abs/1706.03762)

Please complete the `MultiHeadAttention` class in `model.py` using the `SingleHeadAttention` class implemented earlier.

In [ ]:
model = MultiHeadAttention(MiniGPTConfig.embed_dim, MiniGPTConfig.num_heads)
tests.check_multiheadattention(model, path_to_gpt_tester, device)

'TEST CASE PASSED!!!'

### Feed Forward Layer (5 points)

As discussed in lecture, the attention layer is completely linear, in order to add some non-linearity we add a feed forward layer. The feed forward layer is a simple two layer MLP with a GeLU activation in between.

Please complete the `FeedForwardLayer` class in `model.py`

In [ ]:
model = FeedForwardLayer(MiniGPTConfig.embed_dim)
tests.check_feedforward(model, path_to_gpt_tester, device)

'TEST CASE PASSED!!!'

### LayerNorm (10 points)

We will now implement the layer normalization layer. Layernorm is used across the model to normalize the activations of the previous layer. Recall that the equation for layernorm is given as:

\begin{equation}
\text{LayerNorm}(x) = \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}} \odot \gamma + \beta
\end{equation}

With the learnable parameters $\gamma$ and $\beta$.

Remember that unlike batchnorm we compute statistics across the feature dimension and not the batch dimension, hence we do not need to keep track of running averages.

Please complete the `LayerNorm` class in `model.py`

In [ ]:
model = LayerNorm(MiniGPTConfig.embed_dim)
tests.check_layernorm(model, path_to_gpt_tester, device)

'TEST CASE PASSED!!!'

### Transformer Layer (15 points)

We have now implemented all the components of the transformer layer. We will now put it all together to create a transformer layer. The transformer layer consists of a multi head attention layer, a feed forward layer and two layer norm layers.

Please use the following order for each component (Varies slightly from the original attention paper):
1. LayerNorm
2. MultiHeadAttention
3. LayerNorm
4. FeedForwardLayer

Remember that the transformer layer also has residual connections around each sublayer.

The below figure shows the structure of the transformer layer you are required to implement.

![prenorm_transformer](./Images/Prenorm.png)

Image Credit : [CogView](https://arxiv.org/pdf/2105.13290)

Implement the `TransformerLayer` class in `model.py`

In [ ]:
model =  TransformerLayer(MiniGPTConfig.embed_dim, MiniGPTConfig.num_heads)
tests.check_transformer(model, path_to_gpt_tester, device)

'TEST CASE PASSED!!!'

### Putting it all together : MiniGPT (15 points)

We are now ready to put all our layers together to build our own MiniGPT!

The MiniGPT model consists of an embedding layer, a positional encoding layer and a stack of transformer layers. The output of the transformer layer is passed through a linear layer (called head) to get the final output logits. Note that in our implementation we will use [weight tying](https://arxiv.org/abs/1608.05859) between the embedding layer and the final linear layer. This allows us to save on parameters and also helps in training.

Implement the `MiniGPT` class in `model.py`

In [ ]:
model = MiniGPT(MiniGPTConfig)
tests.check_miniGPT(model, path_to_gpt_tester, device)

'TEST CASE PASSED!!!'

### Attempt at training the model (5 points)

We will now attempt to train the model on the text data. We will use the same text data as before. If needed, you can scale down the model parameters in the config file to a smaller value to make training feasible.

Use the same training script we built for the Bigram model to train the MiniGPT model. If you implemented it correctly it should work just out of the box!

**NOTE** : We will not be able to train the model to completion in this assignment. Unfortunately, without access to a relatively powerful GPU, training a large enough model to see good generation is not feasible. However, you should be able to see the loss decreasing over time. <span style="color:red">To get full points for this section it is sufficient to show that the loss is decreasing over time</span>. You do not need to run this for more than 5000 iterations or 1 hour of training.

In [ ]:
from train import solver

In [ ]:
solver(model_name="minigpt")

### Train and Valid Plots


** Show the training and validation loss plots **

### Generation (5 points)


Perform generation with the MiniGPT model that you trained. Generate a mini story using the same seed sentence.

    `"once upon a time"`

In [ ]:
# TODO: Specify the path to your trained model
model_path = "/home/zugho/MiniGPTStudentVersion/models/minigpt/mini_model_checkpoint_250000.pt"
model = MiniGPT(MiniGPTConfig)
tokenizer = tiktoken.get_encoding("gpt2")
model.load_state_dict(torch.load(model_path)["model_state_dict"])

<All keys matched successfully>

In [ ]:
model.to(device)
gen_sent = "Once upon a time"
gen_tokens = torch.tensor(tokenizer.encode(gen_sent)).unsqueeze(0)
print("Generating text starting with:", gen_tokens.shape)
gen_tokens = gen_tokens.to(device)
model.eval()
print(
    tokenizer.decode(
        model.generate(gen_tokens, max_new_tokens=200).squeeze().tolist()
    )
)

Generating text starting with: torch.Size([1, 4])
Once upon a time, there was a little girl named Lily. She loved to read her book and put it on the walls. It was a sad understand why she thought it was long.
"Okay, let's go," Tom and Timmy say. He were going to buy some fruits and had to take you to the farmer. I love to watch. I need to be friends again."
The distance were sad and didn't know what to do. But then, something bad happened and it gave the yard to join them. He ate it with his friend and he remembered how big it was out. But when she went up, she jumped into the grass. The girl and Mia dug a fresh People had aniest toy store. Do you want to play with your toys," his mom replied. From that day on, Timmy's dad was hiding in the forest, there was a war. He lived so much. He was happy to see him or his friends. chase barked the cat and jumped


Please answer the following questions.

1. What can we say about the generated text in terms of grammar and coherence?

The grammar and coherence are a massive leap forward compared to the Bigram model! The MiniGPT successfully generated full, grammatically correct sentences that actually flow logically ("Once upon a time, there was a girl named Lily..."). It maintained the subject ("she") and kept a coherent local narrative. However, you can see it start to lose its global coherence towards the very end ("dug a fresh People had aniest toy store..."), which is completely normal for a model of this small scale.

2. If the model is scaled with more parameters do you expect the GPT model to get substantially better? Why or why not?

Yes, absolutely. Unlike the Bigram model (which is fundamentally limited by its architecture to only look at one previous word), the Transformer architecture is designed to scale. Scaling up parameters (more layers, wider hidden dimensions, more attention heads) gives the self-attention mechanism more "capacity" to learn complex grammatical rules, track long-range dependencies across paragraphs, and build a better internal representation of the text. This is exactly the core principle behind why models like GPT-3 and GPT-4 are so incredibly capable!

### Scaling up the model (5 points)

To show that scale indeed will help the model learn we have trained a scaled up version of the model you just implemented. We will load the weights of this model and generate a mini story using the same seed sentence. Note that if you have implemented the model correctly just scaling the parameters and adding a few bells and whistles to the training script will results in a model like the one we will load now.

In [ ]:
from model import MiniGPT
from config import MiniGPTConfig

In [ ]:
path_to_trained_model = "pretrained_models/best_train_loss_checkpoint.pth"

In [ ]:
ckpt = torch.load(path_to_trained_model, map_location=device) # remove map location if using GPU

In [ ]:
# Set the configs for scaled model
MiniGPTConfig.context_length = 512
MiniGPTConfig.embed_dim = 256
MiniGPTConfig.num_heads = 16
MiniGPTConfig.num_layers = 8

In [ ]:
# Load model from checkpoint
model = MiniGPT(MiniGPTConfig)
model.load_state_dict(ckpt["model_state_dict"])

<All keys matched successfully>

In [ ]:
tokenizer = tiktoken.get_encoding("gpt2")

In [ ]:
model.to(device)
gen_sent = "Once upon a time"
gen_tokens = torch.tensor(tokenizer.encode(gen_sent)).unsqueeze(0)
print("Generating text starting with:", gen_tokens.shape)
gen_tokens = gen_tokens.to(device)
model.eval()

# Tell PyTorch to stop tracking gradients for generation!
with torch.no_grad():
    generated_output = model.generate(gen_tokens, max_new_tokens=50).squeeze().tolist()

print(tokenizer.decode(generated_output))

Generating text starting with: torch.Size([1, 4])
Once upon a time, there was a girl named Lily who was very lucky. One day, she saw a big sign that said "No exaggeration". cinematic took a Cube names and learned something new. She was so proud of herself! She gave her brother a big hug


## Bonus (5 points)

The following are some open ended questions that you can attempt if you have time. Feel free to propose your own as well if you have an interesting idea.

1. The model we have implemented is a decoder only model. Can you implement the encoder part as well? This should not be too hard to do since most of the layers are already implemented.

Yes, it is surprisingly easy since we already have the building blocks. The only difference between a Transformer Decoder (which we built) and a Transformer Encoder (like BERT) is the attention mask.

The Decoder uses a causal mask (a lower-triangular matrix) to prevent tokens from "looking ahead" at future tokens during training.

The Encoder uses bidirectional self-attention, meaning it looks at the entire sequence at once.

Implementation: To turn the TransformerLayer into an encoder block, we would simply remove the causal_mask from the SingleHeadAttention class so that the attention matrix calculates the relationships between all words simultaneously. (Note: If we wanted to build a full Encoder-Decoder architecture like T5 or BART, we would also need to add a "Cross-Attention" layer in the decoder where Queries come from the decoder, and Keys/Values come from the encoder).

2. What are some improvements we can add to the training script to make training more efficient and faster? Can you concretely show that the improvements you made help in training the model better?

There are several industry-standard PyTorch optimizations we can drop into train.py to massively speed up training and reduce memory limits:Mixed Precision Training (AMP): By using torch.autocast and torch.cuda.amp.GradScaler, PyTorch will automatically cast certain operations to 16-bit floats (fp16 or bf16) instead of 32-bit floats. This effectively halves our memory footprint and speeds up computation on modern GPUs (like the T4) without hurting model accuracy.Gradient Accumulation: If we want a batch size of 64 but your GPU crashes at 16, we can run 4 forward/backward passes at batch size 16 before calling optimizer.step(). It mathematically simulates a larger batch size.Flash Attention: Instead of manually computing $QK^T$ and applying softmax, we can use PyTorch 2.0's torch.nn.functional.scaled_dot_product_attention. It is a highly fused, memory-optimized kernel that calculates attention much faster than doing the matrix multiplications line-by-line.torch.compile: Wrapping our model in model = torch.compile(model) optimizes the underlying execution graph for the specific GPU.

3. Can you implement a beam search decoder to generate the text instead of greedy decoding? Does this help in generating better text?

Yes. Right now, our generate function uses multinomial sampling (or greedy decoding if we just took the argmax). At each step, it picks a single next token and commits to it.

How to implement it: Instead of keeping 1 sequence, Beam Search tracks the top k (the "beam width") most probable sequences at every step. For each of the k sequences, we predict the next tokens, calculate the new cumulative probabilities, sort them, and keep only the top k winners to move to the next step.

Does it help? Yes and no. It prevents the model from falling into "garden-path" sentences (where a locally good word leads to a grammatical dead-end). However, for creative text generation (like story writing), Beam Search often produces generic, repetitive, and boring text ("I went to the store and went to the store"). It is much better suited for exact tasks like machine translation or code generation.

4. Can you further optimize the model architecture? For example, can you implement [Multi Query Attention](https://arxiv.org/abs/1911.02150) or [Grouped Query Attention](https://arxiv.org/pdf/2305.13245) to improve the model performance?

Yes, implementing MQA or GQA is one of the most effective ways to optimize a GPT model.

The Problem: In standard Multi-Head Attention (MHA), every single attention head has its own Query, Key, and Value weights. During generation, storing all those Keys and Values in memory (the KV Cache) creates a massive memory-bandwidth bottleneck.

MQA (Multi-Query Attention): We keep multiple Query heads, but we only have one shared Key head and one shared Value head.

GQA (Grouped-Query Attention): A middle ground. We divide our Query heads into "groups" (e.g., 8 query heads divided into 2 groups of 4). Each group shares a Key and Value head.

The Result: The model size shrinks slightly, training speed stays roughly the same, but inference (generation) speed skyrockets because the KV cache is a fraction of its original size. Llama-2 and Llama-3 heavily rely on GQA for this exact reason!

For the neatness of the final submission, I just clear the ouput of 2 training cells. The details of training dynamics are avalible here: https://wandb.ai/zhangh2024-university-of-california/dl2_proj3/reports/Bigram-MiniGPT--VmlldzoxNjc4MzY3OA?accessToken=e7ubfsh7swh5kqwxdy8j0myudm2fh0yg3zte21felnkbav70qpn1z00pyqurs8be

For your convenience, I just list core parameters, metrics, and training loss charts as follows:

| Model | Training Loss |  Evaluation Loss |
| --- | --- |--- |
| Bigram | 4.331120491027832 |  3.971784830093384 |
| MiniGPT | 2.7018003463745117 |  2.942306971549988 |

max iteration: 250000

![Bigram Training Loss](./Language_Modelling_pics/image-3.png)
![MiniGPT Training Loss](./Language_Modelling_pics/image-2.png)

OS: Linux-6.8.0-1053-gcp-x86_64-with-glibc2.35

Python version: CPython 3.10.12

System Hardware

CPU count	2

Logical CPU count	4

GPU count	1

GPU type	Tesla T4
